In [1]:
# === Cell 0: Load credentials from Kaggle Secrets (Add-ons -> Secrets) ===  
import os  
from kaggle_secrets import UserSecretsClient  
  
secrets = UserSecretsClient()  
  
# W&B (optional but recommended). If the secret is missing, training still runs offline.  
try:  
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")  
    print("WANDB_API_KEY loaded from Secrets.")  
except Exception:  
    os.environ["WANDB_DISABLED"] = "true"  
    print("No WANDB_API_KEY secret found -> W&B disabled (training continues).")  
  
# GitHub token (only needed if you push from the notebook; not required to produce the .pth)  
try:  
    os.environ["GITHUB_TOKEN"] = secrets.get_secret("GITHUB_TOKEN")  
    print("GITHUB_TOKEN loaded from Secrets.")  
except Exception:  
    print("No GITHUB_TOKEN secret found -> skipping any git push steps.")  
  
# IMPORTANT: nothing below calls input()/getpass(), so Commit runs never hang.

WANDB_API_KEY loaded from Secrets.
GITHUB_TOKEN loaded from Secrets.


In [2]:
%%writefile data_prep.py  
import os, glob, argparse  
import pandas as pd  
from sklearn.model_selection import StratifiedGroupKFold  
  
# --- Only read mounted, read-only data. NEVER kagglehub-download (quota/wipe risk). ---  
INPUT_ROOT = "/kaggle/input/datasets"  
MALIGNANT = {"MEL"}   # melanoma-only, matches backend LABEL_MAP {1: "Melanoma"}  
  
def _index_images(root):  
    """Map image basename (no ext) -> absolute path, for all jpg/png under root."""  
    idx = {}  
    for ext in ("*.jpg", "*.jpeg", "*.png"):  
        for p in glob.glob(os.path.join(root, "**", ext), recursive=True):  
            idx[os.path.splitext(os.path.basename(p))[0]] = p  
    return idx  
  
def load_isic2019():  
    base = f"{INPUT_ROOT}/andrewmvd/isic-2019"  
    gt = pd.read_csv(f"{base}/ISIC_2019_Training_GroundTruth.csv")  
    imgs = _index_images(base)  
    df = pd.DataFrame({  
        "filepath": gt["image"].map(imgs),  
        "label": (gt["MEL"] == 1).astype(int),  
        "group": gt["image"],           # no lesion_id in GT -> per-image group  
        "source": "isic2019",  
    }).dropna(subset=["filepath"])  
    return df  
  
def load_isic2020():  
    base = f"{INPUT_ROOT}/nischaydnk/isic-2020-jpg-224x224-resized"  
    meta = pd.read_csv(f"{base}/train-metadata.csv")  
    imgs = _index_images(base)  
    df = pd.DataFrame({  
        "filepath": meta["isic_id"].map(imgs),  
        "label": meta["target"].astype(int),  
        "group": meta["patient_id"].fillna(meta["isic_id"]),  
        "source": "isic2020",  
    }).dropna(subset=["filepath"])  
    return df  
  
def load_padufes():  
    base = f"{INPUT_ROOT}/mahdavi1202/skin-cancer"  
    meta = pd.read_csv(f"{base}/metadata.csv")  
    imgs = _index_images(base)  
    df = pd.DataFrame({  
        "filepath": meta["img_id"].map(lambda s: imgs.get(os.path.splitext(str(s))[0])),  
        "label": meta["diagnostic"].astype(str).str.upper().isin(MALIGNANT).astype(int),  
        "group": meta["patient_id"].fillna(meta["lesion_id"]).fillna(meta["img_id"]),  
        "source": "padufes",  
    }).dropna(subset=["filepath"])  
    return df  
  
def main():  
    ap = argparse.ArgumentParser()  
    ap.add_argument("--random-state", type=int, default=42)  
    args = ap.parse_args()  
  
    frames = []  
    for name, fn in [("ISIC2019", load_isic2019), ("ISIC2020", load_isic2020), ("PAD-UFES", load_padufes)]:  
        df = fn()  
        assert len(df) > 0, f"{name}: 0 rows matched — check /kaggle/input layout"  
        print(f"{name}: {len(df)} rows, mal={df.label.mean()*100:.2f}%")  
        frames.append(df)  
  
    merged = pd.concat(frames, ignore_index=True)  
    print(f"MERGED: {len(merged)} rows, mal={merged.label.mean()*100:.2f}%")  
  
    # Patient-level split, leakage-free: 6 folds ~ test 16.7%, then calib from train.  
    sgkf = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=args.random_state)  
    tr_idx, te_idx = next(sgkf.split(merged, merged.label, merged.group))  
    test_df = merged.iloc[te_idx].reset_index(drop=True)  
    trainpool = merged.iloc[tr_idx].reset_index(drop=True)  
  
    sgkf2 = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=args.random_state)  
    tr2, ca2 = next(sgkf2.split(trainpool, trainpool.label, trainpool.group))  
    train_df = trainpool.iloc[tr2].reset_index(drop=True)  
    calib_df = trainpool.iloc[ca2].reset_index(drop=True)  
  
    os.makedirs("data", exist_ok=True)  
    train_df.to_csv("data/train_df.csv", index=False)  
    calib_df.to_csv("data/calib_df.csv", index=False)  
    test_df.to_csv("data/test_df.csv", index=False)  
    print(f"train_df: {len(train_df)} (mal {train_df.label.mean()*100:.2f}%)")  
    print(f"calib_df: {len(calib_df)} (mal {calib_df.label.mean()*100:.2f}%)")  
    print(f"test_df: {len(test_df)} (mal {test_df.label.mean()*100:.2f}%)")  
  
if __name__ == "__main__":  
    main()

Writing data_prep.py


In [3]:
%%writefile model_training.py  
import os, argparse  
import numpy as np, pandas as pd  
import torch, torch.nn as nn  
from torch.utils.data import Dataset, DataLoader  
from torchvision import models, transforms  
from PIL import Image  
from sklearn.metrics import roc_auc_score  
  
TFM = transforms.Compose([  
    transforms.Resize((224, 224)),  
    transforms.ToTensor(),  
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  
])  
  
class DF(Dataset):  
    def __init__(self, csv, train=False):  
        self.df = pd.read_csv(csv)  
        self.train = train  
        self.aug = transforms.Compose([  
            transforms.RandomHorizontalFlip(),  
            transforms.RandomVerticalFlip(),  
            transforms.ColorJitter(0.1, 0.1, 0.1),  
        ])  
    def __len__(self): return len(self.df)  
    def __getitem__(self, i):  
        row = self.df.iloc[i]  
        img = Image.open(row["filepath"]).convert("RGB")  
        if self.train: img = self.aug(img)  
        return TFM(img), torch.tensor([row["label"]], dtype=torch.float32)  
  
def build_backbone(pretrained=True):  
    w = models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None  
    m = models.mobilenet_v2(weights=w)  
    m.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(m.last_channel, 1))  # sigmoid stripped  
    return m  
  
class FocalLoss(nn.Module):  
    def __init__(self, alpha=0.25, gamma=2.0):  
        super().__init__(); self.a, self.g = alpha, gamma  
    def forward(self, logits, y):  
        p = torch.sigmoid(logits)  
        ce = nn.functional.binary_cross_entropy_with_logits(logits, y, reduction="none")  
        pt = p*y + (1-p)*(1-y)  
        at = self.a*y + (1-self.a)*(1-y)  
        return (at * (1-pt)**self.g * ce).mean()  
  
def main():  
    ap = argparse.ArgumentParser()  
    ap.add_argument("--epochs", type=int, default=12)  
    ap.add_argument("--batch-size", type=int, default=64)  
    ap.add_argument("--lr", type=float, default=1e-4)  
    args = ap.parse_args()  
  
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")  
    print("Device:", dev)  
  
    full = pd.read_csv("data/train_df.csv")  
    n_val = int(len(full)*0.15)  
    val = full.sample(n=n_val, random_state=42); tr = full.drop(val.index)  
    tr.to_csv("data/_tr.csv", index=False); val.to_csv("data/_val.csv", index=False)  
    print(f"train={len(tr)} val={len(val)} mal_train={tr.label.mean()*100:.2f}%")  
  
    dl_tr = DataLoader(DF("data/_tr.csv", train=True),  batch_size=args.batch_size,  
                       shuffle=True, num_workers=2, pin_memory=True)  
    dl_va = DataLoader(DF("data/_val.csv", train=False), batch_size=args.batch_size,  
                       shuffle=False, num_workers=2, pin_memory=True)  
  
    # W&B only if the Secret provided a key (Cell 0). Never prompts.  
    use_wandb = bool(os.getenv("WANDB_API_KEY")) and os.getenv("WANDB_DISABLED") != "true"  
    if use_wandb:  
        import wandb  
        wandb.init(project="skin-cancer-absention", job_type="train")  
  
    model = build_backbone(pretrained=True).to(dev)  
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)  
    loss_fn = FocalLoss()  
  
    best_auc = -1.0  
    for ep in range(1, args.epochs+1):  
        model.train(); tot = 0.0  
        for x, y in dl_tr:  
            x, y = x.to(dev), y.to(dev)  
            opt.zero_grad()  
            loss = loss_fn(model(x), y)  
            loss.backward(); opt.step()  
            tot += loss.item()*len(x)  
        model.eval(); ps, ys = [], []  
        with torch.no_grad():  
            for x, y in dl_va:  
                p = torch.sigmoid(model(x.to(dev))).cpu().numpy().ravel()  
                ps += p.tolist(); ys += y.numpy().ravel().tolist()  
        auc = roc_auc_score(ys, ps)  
        print(f"epoch {ep}/{args.epochs}  loss={tot/len(tr):.4f}  val_auc={auc:.4f}")  
        if use_wandb: wandb.log({"epoch": ep, "loss": tot/len(tr), "val_auc": auc})  
        if auc > best_auc:  
            best_auc = auc  
            torch.save(model.state_dict(), "mobilenetv2_isic.pth")  
            print(f"  ↑ new best AUC {auc:.4f} (checkpointed)")  
    print(f"Saved best backbone (val_auc={best_auc:.4f}) -> mobilenetv2_isic.pth")  
    if use_wandb: wandb.finish()  
  
if __name__ == "__main__":  
    main()

Writing model_training.py


In [4]:
%%writefile calibration.py  
import pandas as pd, torch, torch.nn as nn  
from torch.utils.data import DataLoader  
from torchvision import models, transforms  
from PIL import Image  
from torch.utils.data import Dataset  
  
TFM = transforms.Compose([  
    transforms.Resize((224, 224)), transforms.ToTensor(),  
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),  
])  
  
class DF(Dataset):  
    def __init__(self, csv): self.df = pd.read_csv(csv)  
    def __len__(self): return len(self.df)  
    def __getitem__(self, i):  
        r = self.df.iloc[i]  
        return TFM(Image.open(r["filepath"]).convert("RGB")), torch.tensor([r["label"]], dtype=torch.float32)  
  
class ModelWithTemperature(nn.Module):  
    """Mirrors backend/app.py: MobileNetV2 + Dropout/Linear head (no sigmoid) + temperature param."""  
    def __init__(self, temperature=1.0):  
        super().__init__()  
        b = models.mobilenet_v2(weights=None)  
        b.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(b.last_channel, 1))  
        self.model = b  
        self.temperature = nn.Parameter(torch.ones(1) * float(temperature))  
    def forward(self, x):  
        return self.model(x) / self.temperature.clamp_min(1e-6)  
  
def main():  
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")  
    wrapper = ModelWithTemperature().to(dev)  
    # load the trained backbone into wrapper.model.*  
    state = torch.load("mobilenetv2_isic.pth", map_location=dev)  
    wrapper.model.load_state_dict(state)  
  
    dl = DataLoader(DF("data/calib_df.csv"), batch_size=64, shuffle=False, num_workers=2)  
    logits, labels = [], []  
    wrapper.eval()  
    with torch.no_grad():  
        for x, y in dl:  
            logits.append(wrapper.model(x.to(dev)).cpu()); labels.append(y)  
    logits = torch.cat(logits); labels = torch.cat(labels)  
  
    T = nn.Parameter(torch.ones(1))  
    opt = torch.optim.LBFGS([T], lr=0.01, max_iter=100)  
    bce = nn.BCEWithLogitsLoss()  
    def closure():  
        opt.zero_grad(); loss = bce(logits / T.clamp_min(1e-6), labels); loss.backward(); return loss  
    opt.step(closure)  
    wrapper.temperature.data = T.data.clone().to(dev)  
    print(f"Fitted temperature T = {T.item():.4f}")  
  
    torch.save(wrapper.state_dict(), "mobilenetv2_calibrated.pth")  
    print("Saved calibrated checkpoint -> mobilenetv2_calibrated.pth")  
  
if __name__ == "__main__":  
    main()

Writing calibration.py


In [5]:
%%writefile evaluate_calibrated_model.py  
import pandas as pd, numpy as np, torch, torch.nn as nn  
from torch.utils.data import Dataset, DataLoader  
from torchvision import models, transforms  
from PIL import Image  
from sklearn.metrics import roc_auc_score, average_precision_score  
  
TFM = transforms.Compose([  
    transforms.Resize((224,224)), transforms.ToTensor(),  
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),  
])  
class DF(Dataset):  
    def __init__(self, csv): self.df = pd.read_csv(csv)  
    def __len__(self): return len(self.df)  
    def __getitem__(self, i):  
        r = self.df.iloc[i]  
        return TFM(Image.open(r["filepath"]).convert("RGB")), float(r["label"])  
  
class ModelWithTemperature(nn.Module):  
    def __init__(self, temperature=1.0):  
        super().__init__()  
        b = models.mobilenet_v2(weights=None)  
        b.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(b.last_channel, 1))  
        self.model = b  
        self.temperature = nn.Parameter(torch.ones(1)*float(temperature))  
    def forward(self, x): return self.model(x)/self.temperature.clamp_min(1e-6)  
  
def ece(probs, labels, bins=15):  
    probs, labels = np.array(probs), np.array(labels)  
    edges = np.linspace(0,1,bins+1); e = 0.0  
    for i in range(bins):  
        m = (probs>edges[i]) & (probs<=edges[i+1])  
        if m.sum()==0: continue  
        e += (m.mean()) * abs(labels[m].mean() - probs[m].mean())  
    return e  
  
def main():  
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")  
    print("Eval on", dev)  
    m = ModelWithTemperature().to(dev)  
    state = torch.load("mobilenetv2_calibrated.pth", map_location=dev)  
    m.load_state_dict(state, strict=True)     # strict: verifies key alignment with backend  
    print("keys align. Embedded T =", m.temperature.item())  
    m.eval()  
  
    dl = DataLoader(DF("data/test_df.csv"), batch_size=64, shuffle=False, num_workers=2)  
    ps, ys = [], []  
    with torch.no_grad():  
        for x, y in dl:  
            p = torch.sigmoid(m(x.to(dev))).cpu().numpy().ravel()  
            ps += p.tolist(); ys += list(np.array(y).ravel())  
    ps, ys = np.array(ps), np.array(ys)  
    pred = (ps >= 0.5).astype(int)  
    tp = ((pred==1)&(ys==1)).sum(); fn = ((pred==0)&(ys==1)).sum()  
    tn = ((pred==0)&(ys==0)).sum(); fp = ((pred==1)&(ys==0)).sum()  
    print(f"AUC={roc_auc_score(ys,ps):.5f} PR-AUC={average_precision_score(ys,ps):.5f} ECE={ece(ps,ys):.5f}")  
    print(f"Sens={tp/(tp+fn)*100:.2f}% Spec={tn/(tn+fp)*100:.2f}% PPV={tp/(tp+fp+1e-9)*100:.2f}%")  
    print(f"\nSet in backend env:\nMODEL_AUC={roc_auc_score(ys,ps):.5f}\nMODEL_ECE={ece(ps,ys):.5f}")  
  
if __name__ == "__main__":  
    main()

Writing evaluate_calibrated_model.py


In [6]:
# === Cell 5: run everything, verify the .pth exists, log a W&B artifact ===  
!python data_prep.py --random-state 42  
!python model_training.py --epochs 12 --batch-size 64  
!python calibration.py  
!python evaluate_calibrated_model.py  
  
# Confirm the checkpoint is really on disk BEFORE you close the tab  
!ls -lh /kaggle/working/mobilenetv2_calibrated.pth  
  
# Safety net: log the checkpoint as a W&B artifact so a dead session never costs you the run again.  
import os  
if os.getenv("WANDB_API_KEY") and os.getenv("WANDB_DISABLED") != "true":  
    import wandb  
    run = wandb.init(project="skin-cancer-absention", job_type="checkpoint")  
    art = wandb.Artifact("mobilenetv2_calibrated", type="model")  
    art.add_file("/kaggle/working/mobilenetv2_calibrated.pth")  
    run.log_artifact(art); run.finish()  
    print("Logged checkpoint to W&B Artifacts (downloadable later, no rerun needed).")

ISIC2019: 25331 rows, mal=17.85%
ISIC2020: 33126 rows, mal=1.76%
PAD-UFES: 2298 rows, mal=2.26%
MERGED: 60755 rows, mal=8.49%
train_df: 42229 (mal 8.38%)
calib_df: 8202 (mal 8.88%)
test_df: 10324 (mal 8.63%)
Device: cuda
train=35895 val=6334 mal_train=8.34%
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: hdd5ps (hdd5ps-university-of-virginia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260907_002128-azg0ysib
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run charmed-gorge-5
wandb: ⭐️ View project at https://wandb.ai/hdd5ps-university-of-virginia/skin-cancer-absention
wandb: 🚀 View run at https://wandb.ai/hdd5ps-university-of-virginia/skin-cancer-absention/runs/azg0ysib
Downloading: "https://down

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: hdd5ps (hdd5ps-university-of-virginia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260907_015035-wothl2jx
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run legendary-aardvark-6
wandb: ⭐️ View project at https://wandb.ai/hdd5ps-university-of-virginia/skin-cancer-absention
wandb: 🚀 View run at https://wandb.ai/hdd5ps-university-of-virginia/skin-cancer-absention/runs/wothl2jx
wandb: updating run metadata; uploading artifact mobilenetv2_calibrated; uploading summary
wandb: uploading artifact mobilenetv2_calibrated
wandb: uploading data
wandb: 🚀 View run legendary-aardvark-6 at: https://wandb.ai/hdd5ps-university-of-virginia/skin-cancer-absention/runs/wothl2jx
wandb: ⭐️ View project at: https://wandb.ai/hdd5ps-university-of-

Logged checkpoint to W&B Artifacts (downloadable later, no rerun needed).


In [7]:
import torch, torch.nn as nn  
from torchvision import models  
  
class ModelWithTemperature(nn.Module):  
    def __init__(self, temperature=1.0):  
        super().__init__()  
        b = models.mobilenet_v2(weights=None)  
        b.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(b.last_channel, 1))  
        self.model = b  
        self.temperature = nn.Parameter(torch.ones(1) * float(temperature))  
  
state = torch.load("mobilenetv2_calibrated.pth", map_location="cpu")  
m = ModelWithTemperature()  
missing, unexpected = m.load_state_dict(state, strict=False)  
assert not missing and not unexpected, f"MISMATCH missing={missing} unexpected={unexpected}"  
print("✅ keys align with backend. Embedded T =", state["temperature"].item())  
!ls -la mobilenetv2_calibrated.pth

✅ keys align with backend. Embedded T = 0.792801022529602
-rw-r--r-- 1 root root 9153688 Sep  7 01:49 mobilenetv2_calibrated.pth


In [8]:
import torch, torch.nn as nn, numpy as np, pandas as pd, cv2  
from torchvision import models, transforms  
from sklearn.metrics import confusion_matrix, roc_auc_score  
  
device = "cuda" if torch.cuda.is_available() else "cpu"  
  
# --- 1. Rebuild the SAME wrapper as backend ModelWithTemperature ---  
class ModelWithTemperature(nn.Module):  
    def __init__(self, temperature=1.0):  
        super().__init__()  
        backbone = models.mobilenet_v2(weights=None)  
        backbone.classifier = nn.Sequential(  
            nn.Dropout(p=0.2),  
            nn.Linear(backbone.last_channel, 1),  # sigmoid stripped for logit inference  
        )  
        self.model = backbone  
        self.temperature = nn.Parameter(torch.ones(1) * float(temperature))  
    def forward(self, x):  
        return self.model(x) / self.temperature.clamp_min(1e-6)  
  
def build_calibrated_model():  
    return ModelWithTemperature()  
  
# --- 2. Load checkpoint strictly (ZERO missing/unexpected keys) ---  
model = build_calibrated_model().to(device)  
state = torch.load("mobilenetv2_calibrated.pth", map_location=device)  
model.load_state_dict(state, strict=True)  
model.eval()  
print("Embedded T =", model.temperature.item())  
  
# --- 3. Recompute y_true / y_prob on the held-out test set ---  
tf = transforms.Compose([  
    transforms.ToPILImage(),  
    transforms.Resize((224, 224)),  
    transforms.ToTensor(),  
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),  
])  
test_df = pd.read_csv("data/test_df.csv")   # adjust path if different  
PATH_COL, LABEL_COL = "filepath", "label"   # adjust to your data_prep column names  
  
y_true, y_prob = [], []  
with torch.no_grad():  
    for _, row in test_df.iterrows():  
        img = cv2.cvtColor(cv2.imread(row[PATH_COL]), cv2.COLOR_BGR2RGB)  
        x = tf(img).unsqueeze(0).to(device)  
        p = torch.sigmoid(model(x)).item()   # temperature already applied inside forward  
        y_prob.append(p); y_true.append(int(row[LABEL_COL]))  
y_true, y_prob = np.array(y_true), np.array(y_prob)  
print("AUC =", roc_auc_score(y_true, y_prob), " n =", len(y_true), " mal% =", y_true.mean()*100)  
  
# --- 4. Threshold sweep ---  
for t in [0.05,0.10,0.15,0.20,0.25,0.30,0.40,0.50]:  
    tn,fp,fn,tp = confusion_matrix(y_true, (y_prob>=t).astype(int)).ravel()  
    sens = tp/(tp+fn); spec = tn/(tn+fp); ppv = tp/(tp+fp+1e-9)  
    print(f"t={t:.2f}  Sens={sens:.3f}  Spec={spec:.3f}  PPV={ppv:.3f}")

Embedded T = 0.792801022529602
AUC = 0.9383889188122553  n = 10324  mal% = 8.630375823324293
t=0.05  Sens=0.928  Spec=0.743  PPV=0.254
t=0.10  Sens=0.866  Spec=0.846  PPV=0.347
t=0.15  Sens=0.824  Spec=0.898  PPV=0.434
t=0.20  Sens=0.763  Spec=0.928  PPV=0.500
t=0.25  Sens=0.714  Spec=0.949  PPV=0.568
t=0.30  Sens=0.637  Spec=0.964  PPV=0.625
t=0.40  Sens=0.552  Spec=0.983  PPV=0.750
t=0.50  Sens=0.469  Spec=0.991  PPV=0.836
